In [ ]:
%load_ext autoreload
%autoreload 2

from src import GraspAnalysisResults, GraspRegion, InitializationAngle
from src.utils import GraspAnalysisUtils
from pathlib import Path
import numpy as np


In [3]:
# Find folder containing CSV files for analysis
data_folder = GraspAnalysisUtils.select_folder()


Selected folder: C:/Users/rjs15/Documents/FY26/Grasping/Grasp_Strength_Python/data/robotiq_cylinder


In [4]:
# Check if there as csv files in the chosen directory
dir_path = Path(data_folder)
if any(file.suffix.lower() == ".csv" for file in dir_path.iterdir() if file.is_file()):
    files = [str(file) for file in dir_path.glob("*.csv")]
    print("CSV files located")
else:
    raise RuntimeError("No CSV files found in chosen folder")

CSV files located


In [5]:
# Plot Dialog Selection
chosen_plots = GraspAnalysisUtils.select_desired_plots()
print(f"User wants to create: {chosen_plots}")

User wants to create: []


In [6]:
# Assignment of variables
SAMPLE_SIZE = 200
OFFSET = 2000
MIN_FORCE_THRESHOLD = 10

In [7]:
# Performing Analysis

for file_path in files:

    result = GraspAnalysisResults
    full_path = Path(file_path)

    result.filename = full_path.name

    force_data = GraspAnalysisUtils.load_and_preprocess_data(file_path)

    grasps = GraspAnalysisUtils.detect_grasp_regions(force_data, SAMPLE_SIZE, MIN_FORCE_THRESHOLD)

    result.number_of_grasps = len(grasps)

    result.rolling_avg = []
    result.rolling_std = []
    result.rolling_median = []

    if result.number_of_grasps == 0:
        Warning(f"No grasps detected in file: {result.filename}")
        continue

    avg_forces = []
    for grasp in grasps:
        grasp = GraspAnalysisUtils.calculate_grasp_force(force_data[grasp.start_idx:grasp.end_idx], grasp, OFFSET)

        avg_forces.append(grasp.avg_force)
        result.rolling_avg.append(np.mean(avg_forces))
        result.rolling_std.append(np.std(avg_forces))
        result.rolling_median.append(np.median(avg_forces))

    # Calculate Statistics 
    result = GraspAnalysisUtils.calculate_grasp_statistics(result, grasps)

    result = GraspAnalysisUtils.calculate_required_samples(result)

    GraspAnalysisUtils.report_results(result, grasps)

    GraspAnalysisUtils.create_grasp_plots(chosen_plots, force_data, result, grasps)

Filename: RobotIQ_60SFMA_50Cycles_Cyl.csv

The average strength of a grasp is 208.8 N
The standard deviation for grasp strength is 5.22 N
Coefficient of variation: 2.5
95% confidence interval: [(207.31, 210.28)]
95% prediction interval: [(198.21, 219.38)]

Average peak force: 231.13 N
Number of grasps detected: 50

Margin of error: 1.45 N
Recommended number of samples to collect: 24.0
Recommended number of samples (with buffer): 35.0

95% Confidence interval for standard deviation: [4.36, 6.5]
Recommended number of samples to collect based on standard deviation: 17


Filename: RobotIQ_70SFMA_250Cycles_Cyl.csv

The average strength of a grasp is 168.49 N
The standard deviation for grasp strength is 5.42 N
Coefficient of variation: 3.21
95% confidence interval: [(167.81, 169.16)]
95% prediction interval: [(157.8, 179.17)]

Average peak force: 180.48 N
Number of grasps detected: 252

Margin of error: 0.67 N
Recommended number of samples to collect: 40.0
Recommended number of samples (with